# Generalised Linear Models

### [Neil D. Lawrence](http://inverseprobability.com), University of

Cambridge

### 2025-09-10

$$
$$

<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

## ML Foundations Course Notebook Setup

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_mlfc/includes/mlfc-notebook-setup.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_mlfc/includes/mlfc-notebook-setup.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We install some bespoke codes for creating and saving plots as well as
loading data sets.

In [ ]:
%%capture
%pip install notutils
%pip install pods
%pip install mlai

In [ ]:
import notutils
import pods
import mlai
import mlai.plot as plot

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 22})

<!--setupplotcode{import seaborn as sns
sns.set_style('darkgrid')
sns.set_context('paper')
sns.set_palette('colorblind')}-->

In [ ]:
%pip install statsmodels

## Review

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/generalised-linear-models.gpp.markdown" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/generalised-linear-models.gpp.markdown', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We introduced machine learning as a way to extract knowledge from data
to make predictions through a prediction function and an objective
function. We looked at a simple example of predicting whether someone
would buy a jumper based on their age and latitude, *using logistic
regression* to model the log-odds of purchase. This highlighted how
machine learning can codify predictions through mathematical functions.
This is an example of a broader approach known as *generalised linear
models*.

When taking a probabilistic approach to supervised learning we’re
interested in predicting a class label, $y_i$, given an input,
$\mathbf{ x}_i$. That’s represented probabilisticially as
$p(y_i|\mathbf{ x}_i)$. We can derive this conditional distribution
through either (1) modelling the joint distribution,
$p(\mathbf{ y}, \mathbf{X})$ and then dividing by the marginal
distribution of the inputs, $p(\mathbf{X})$ , or (2) focusing
specifically on modeling the conditional density,
$p(\mathbf{ y}|\mathbf{X})$, that directly answers our prediction
question. In the *generalised linear model* we choose the second
approach.

As we move to generalised linear models like logistic regression, we’ll
see how directly modeling the conditional density
$p(\mathbf{ y}|\mathbf{X})$ can provide more flexibility in our modeling
assumptions, while still allowing us to make the specific predictions we
need.

## Linear Regression with `statsmodels`

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/linear-regression-statsmodels.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/linear-regression-statsmodels.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In linear regression, we model the relationship between a continuous
response variable $y_i$ and input variables $\mathbf{ x}_i$ through a
linear function with Gaussian noise:

$$y_i = f(\mathbf{ x}_i) + \epsilon_i$$

where
$f(\mathbf{ x}_i) = \mathbf{ w}^\top\mathbf{ x}_i = \sum_{j=1}^D w_jx_{i,j}$
and $\epsilon_i \sim \mathcal{N}\left(0,\sigma^2\right)$

This gives us a probabilistic model:

$$p(y_i|\mathbf{ x}_i) = \gaussianDist{\mathbf{ w}^\top\mathbf{ x}_i}{\sigma^2}$$

The key components are:

-   $y_i$ is the target/response variable we want to predict
-   $\mathbf{ x}_i$ contains the input features/explanatory variables  
-   $\mathbf{ w}$ contains the parameters/coefficients we learn
-   $\epsilon_i$ represents random Gaussian noise with variance
    $\sigma^2$

For the full dataset, we can write this in matrix form:

$$\mathbf{ y}= \mathbf{X}\mathbf{ w}+ \boldsymbol{ \epsilon}$$

where $\mathbf{ y}= [y_1,\ldots,y_N]^\top$, $\mathbf{X}$ contains the
input vectors as rows, and
$\boldsymbol{ \epsilon}\sim \mathcal{N}\left(\mathbf{0},\sigma^2\mathbf{I}\right)$.

The expected value of our prediction is:

$$\mathbb{E}[y_i|\mathbf{ x}_i] = \mathbf{ w}^\top\mathbf{ x}_i$$

This linear model forms the foundation for generalized linear models
like logistic regression, where we’ll adapt the model for classification
by transforming the output through a non-linear function.

In [ ]:
import statsmodels.api as sm
import pods

In [ ]:
# Demo of linear regression using python statsmodels.
data = pods.datasets.olympic_marathon_men()
x = data['X']
y = data['Y']
# Add constant term to design matrix
x = sm.add_constant(x)
model = sm.OLS(y, x)
results = model.fit()
results.summary()

The statsmodels summary provides several key diagnostic measures that
help us evaluate our model fit and identify potential areas for
improvement. Since we’re working with one-dimensional data (year vs
time), we can visualize everything easily to complement these
statistical measures.

The model fit statistics show a moderately strong fit, with an R-squared
of 0.744 indicating that our model explains 74.4% of the variance in the
data. The adjusted R-squared of 0.733 confirms this isn’t just due to
overfitting. The very low F-statistic p-value (7.40e-09) confirms the
model’s overall significance. The AIC (10.08) and BIC (12.67) values
will be useful when we compare this model against alternative
specifications we might try.

Looking at the model parameters, we see a coefficient of -0.013 for our
predictor, with a small standard error (0.002). The t-statistic of
-8.515 and p-value of 0.000 indicate this effect is highly significant.
The 95% confidence interval \[-0.016, -0.010\] gives us good confidence
in our estimate. The negative coefficient confirms the expected downward
trend in marathon times over the years.

However, the residual diagnostics suggest several potential issues we
should investigate:

1.  The Durbin-Watson statistic (1.110) indicates positive
    autocorrelation in the residuals, though not as severe as we might
    expect. This suggests we might want to:

    -   Consider time series modeling approaches
    -   Add polynomial terms to capture non-linear trends
    -   Investigate if there are distinct “eras” in marathon times

2.  The highly significant Jarque-Bera test (p-value 7.67e-12) tells us
    our residuals aren’t normally distributed. The skew (1.929) and
    kurtosis (8.534) values show the distribution is strongly
    right-skewed with very heavy tails. We might want to:

    -   Look for outliers or influential points
    -   Consider robust regression techniques
    -   Try transforming our response variable

3.  The large condition number (1.08e+05) suggests potential numerical
    instability or multicollinearity issues. While less concerning with
    single-predictor models, we should:

    -   Consider centering and scaling our predictor
    -   Watch for numerical precision issues
    -   Be cautious when extending to multiple predictors

The beauty of having one-dimensional data is that we can plot everything
to visually confirm these statistical findings. A scatter plot with our
fitted line will help us:

-   Visually assess the linearity assumption
-   Identify potential outliers
-   Spot any systematic patterns in the residuals
-   See if the relationship makes practical sense in terms of marathon
    performance over time

This visual inspection, combined with our statistical diagnostics, will
guide our next steps in improving the model.

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
fig, ax = plt.subplots(figsize=mlai.plot.big_wide_figsize)
ax.plot(x[:, 1], y, '.')

# Plot the fitted line
ax.plot(x[:, 1], results.predict(x), '-')

ax.set_xlabel('Year')
ax.set_ylabel('Time')
ax.set_xlim(1890, 2030)
plt.show()
mlai.write_figure("linear-regression-olympic-marathon-men-statsmodels.svg", directory="./data-science")

Looking at our plot and model diagnostics, we can now better understand
the large condition number (1.08e+05) in our results. This high value
likely stems from using raw year values (e.g., 1896, 1900, etc.) as our
predictor variable. Such large numbers can lead to numerical instability
in the computations.

To address this, we could consider:

-   Centering the years around their mean
-   Scaling the years to a smaller range (e.g., 0-1)
-   Using years since the first Olympics (e.g., 0, 4, 8, etc.)

Any of these transformations would help reduce the condition number
while preserving the underlying relationship in our data. The
coefficients would change, but the fitted values and overall model
quality would remain the same.

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//data-science/linear-regression-olympic-marathon-men-statsmodels.svg.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Linear regression fit to Olympic marathon men’s times using
`statsmodels`.</i>

The plot reveals several key features that help explain our diagnostic
statistics:

-   The 1904 St. Louis Olympics appears as a clear outlier, contributing
    to the non-normal residuals (Jarque-Bera p=0.00432) and right-skewed
    distribution (skew=1.385)
-   We can observe distinct regimes in the data:
    -   Rapid improvement in times pre-WWI
    -   Disruption and variation during the war years
    -   More steady, consistent progress post-WWII
-   These regime changes help explain the strong positive
    autocorrelation (Durbin-Watson=0.242) in our residuals
-   While our high R-squared (0.972) captures the overall downward
    trend, these features suggest we could improve the model by adding
    additional features:
    -   Polynomial terms to capture non-linear trends
    -   Indicator variables for different time periods
    -   Interaction terms between features
    -   Variables accounting for external factors like temperature or
        course conditions

To incorporate multiple features into our model, we need a systematic
way to organize this additional information. This brings us to the
concept of the design matrix.

### Design Matrix

The design matrix, often denoted as $\boldsymbol{ \Phi}$, is a key
component of a statistical model. It organizes our feature data in a
structured way that facilitates model fitting and analysis. Each row of
the design matrix represents a single observation or data point, while
each column represents a different feature or predictor variable.

For $n$ observations and $p$ features, the design matrix takes the form:

$$\boldsymbol{ \Phi}= \begin{bmatrix} 
x_{11} & x_{12} & \cdots & x_{1p} \\
x_{21} & x_{22} & \cdots & x_{2p} \\
\vdots & \vdots & \ddots & \vdots \\
x_{n1} & x_{n2} & \cdots & x_{np}
\end{bmatrix}$$

For example, if we’re predicting house prices, each row might represent
a different house, with columns for features like:

-   Square footage
-   Number of bedrooms  
-   Year built
-   Lot size

The design matrix provides a compact way to represent all our feature
data and is used directly in model fitting. When we write our linear
model as
$\mathbf{ y}= \boldsymbol{ \Phi}\mathbf{ w}+ \boldsymbol{ \epsilon}$,
the design matrix $\boldsymbol{ \Phi}$ is multiplied by our parameter
vector $\mathbf{ w}$ to generate predictions.

The design matrix often includes a column of 1s to account for the
intercept term in our model. This allows us to write the model in matrix
form without explicitly separating out the intercept term.

In [ ]:
import statsmodels.api as sm
import pods
import numpy as np

In [ ]:
# Demo of additional features with interactions regression usying python statsmodels.
data = pods.datasets.olympic_marathon_men()
x = data['X']
y = data['Y']

# Scale the year to avoid numerical issues
x_scaled = (x - 1900) / 100  # Center around 1900 and scale to century units

# Add to design matrix indicator variable for pre-1914
x_aug = np.hstack([x_scaled, (x[:, 0] < 1914).astype(np.float64)[:, np.newaxis]])

# Add to design matrix indicator variable for 1914-1945
x_aug = np.hstack([x_aug, ((x[:, 0] >= 1914) & (x[:, 0] <= 1945)).astype(np.float64)[:, np.newaxis]])

# Add to design matrix indicator variable for post-1945
x_aug = np.hstack([x_aug, (x[:, 0] > 1945).astype(np.float64)[:, np.newaxis]])

# Add product terms that multiply the scaled year and the indicator variables.
x_aug = np.hstack([x_aug, x_scaled[:, 0:1] * x_aug[:, 1:2], x_scaled[:, 0:1] * x_aug[:, 2:3]])

# Add constant term to design matrix
x_aug = sm.add_constant(x_aug)

# Do the linear fit
model = sm.OLS(y, x_aug)
results = model.fit()
results.summary()

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
fig, ax = plt.subplots(figsize=mlai.plot.big_wide_figsize)
ax.plot(x[:, 0], y, '.')

# Plot the fitted line
ax.plot(x[:, 0], results.predict(x_aug), '-')

ax.set_xlabel('Year')
ax.set_ylabel('Time')
ax.set_xlim(1890, 2030)
plt.show()
mlai.write_figure("linear-regression-olympic-marathon-men-augmented-statsmodels.svg", directory="./data-science")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//data-science/linear-regression-olympic-marathon-men-augmented-statsmodels.svg.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Polynomial regression fit to Olympic marathon men’s times
using `statsmodels`.</i>

The augmented model with interactions shows a significant improvement in
fit compared to the simpler linear model, with an R-squared value of
0.870 (adjusted R-squared of 0.839). This indicates that about 87% of
the variance in marathon times is explained by our model.

The model includes several key components:

-   A base time trend (x1 coefficient: -0.6737)
-   Indicator variables for different historical periods (pre-1914,
    1914-1945, post-1945)
-   Interaction terms between the time trend and these periods

The coefficients reveal interesting patterns:

-   The pre-1914 period shows a significant positive effect (x2: 1.5506,
    p\<0.001)
-   The wartime period 1914-1945 also shows a positive effect (x3:
    0.7982, p\<0.05)
-   The post-1945 period has a positive effect (x4: 0.6883, p\<0.01)
-   The interaction terms (x5, x6) suggest different rates of
    improvement in different periods, though these are less
    statistically significant

However, there are some concerns:

1.  The very high condition number (2.79e+16) suggests serious
    multicollinearity issues
2.  The Jarque-Bera test (p\<0.001) indicates non-normal residuals
3.  There’s significant skewness (2.314) and kurtosis (10.325) in the
    residuals

Despite these statistical issues, the model captures the major trends in
marathon times across different historical periods better than a simple
linear regression would.

## Logistic Regression with `statsmodels`

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-statsmodels.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-statsmodels.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In logistic regression, we model the relationship between a binary
response variable $y_i \in \{0,1\}$ and input variables $\mathbf{ x}_i$
using the logistic function. Unlike linear regression, we cannot
directly model the probability using a linear function since
probabilities must lie between 0 and 1.

The logistic regression model uses the sigmoid function to map any
real-valued input to the range \[0,1\]:

$$p(y_i = 1|\mathbf{ x}_i) = \sigma(\mathbf{ w}^\top\mathbf{ x}_i) = \frac{1}{1 + \exp(-\mathbf{ w}^\top\mathbf{ x}_i)}$$

where $\sigma(\cdot)$ is the sigmoid function and
$\mathbf{ w}^\top\mathbf{ x}_i = \sum_{j=1}^D w_jx_{i,j}$ is the linear
predictor.

The key components are: - $y_i \in \{0,1\}$ is the binary
target/response variable - $\mathbf{ x}_i$ contains the input
features/explanatory variables  
- $\mathbf{ w}$ contains the parameters/coefficients we learn -
$\sigma(\cdot)$ is the sigmoid activation function that maps
$(-\infty, \infty) \to (0, 1)$

The model assumes that given the features $\mathbf{ x}_i$, the response
$y_i$ follows a Bernoulli distribution:

$$y_i|\mathbf{ x}_i \sim \text{Bernoulli}(\sigma(\mathbf{ w}^\top\mathbf{ x}_i))$$

This gives us the likelihood:

$$p(y_i|\mathbf{ x}_i) = \sigma(\mathbf{ w}^\top\mathbf{ x}_i)^{y_i} \left(1 - \sigma(\mathbf{ w}^\top\mathbf{ x}_i)\right)^{1-y_i}$$

The log-odds (logit) transformation provides a linear relationship:

$$\log\left(\frac{p(y_i = 1|\mathbf{ x}_i)}{1 - p(y_i = 1|\mathbf{ x}_i)}\right) = \mathbf{ w}^\top\mathbf{ x}_i$$

This means the coefficients $\mathbf{ w}$ represent the change in
log-odds for a unit change in the corresponding feature.

In [ ]:
import statsmodels.api as sm
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

In [ ]:
# Demo of logistic regression using python statsmodels.
# Create a synthetic binary classification dataset
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                         n_informative=2, n_clusters_per_class=1, 
                         random_state=42)

# Convert to DataFrame for easier handling
df = pd.DataFrame(X, columns=['feature1', 'feature2'])
df['target'] = y

# Split into train and test sets
indices = np.random.permutation(df.shape[0])
num_train = int(np.ceil(df.shape[0]/2))
train_indices = indices[:num_train]
test_indices = indices[num_train:]

X_train = df[['feature1', 'feature2']].iloc[train_indices]
y_train = df['target'].iloc[train_indices]
X_test = df[['feature1', 'feature2']].iloc[test_indices]
y_test = df['target'].iloc[test_indices]

# Add constant term to design matrix
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# Fit logistic regression model
model = sm.Logit(y_train, X_train_sm)
results = model.fit()
results.summary()

The statsmodels summary for logistic regression provides several
diagnostic measures that help us evaluate our classification model’s
performance and identify potential areas for improvement.

**Model Fit Statistics:** The logistic regression model doesn’t have a
traditional R-squared since we’re dealing with binary outcomes rather
than continuous responses. Instead, we use pseudo R-squared measures:

-   **McFadden’s R-squared**: Compares the log-likelihood of our model
    to a null model with only an intercept. Values between 0.2-0.4
    indicate excellent fit.
-   **Log-Likelihood Ratio (LLR) test**: Tests whether our model is
    significantly better than the null model. A low p-value indicates
    our predictors significantly improve the model.
-   **AIC/BIC**: Help compare different model specifications. Lower
    values indicate better models when comparing alternatives.

**Parameter Interpretation:** The coefficients in logistic regression
represent changes in log-odds: - A positive coefficient means the
feature increases the odds of the positive class - A negative
coefficient means the feature decreases the odds of the positive class  
- The magnitude indicates the strength of the effect - To get odds
ratios, we exponentiate the coefficients: $\exp(\beta_j)$

For example, if a coefficient is 0.693, then $\exp(0.693) = 2.0$,
meaning a one-unit increase in that feature doubles the odds of the
positive outcome.

**Diagnostic Considerations:** Unlike linear regression, logistic
regression has different diagnostic concerns:

1.  **Multicollinearity**: Check condition numbers and correlation
    matrices, just like in linear regression
2.  **Outliers and Influential Points**: Use deviance residuals and
    leverage measures to identify problematic observations
3.  **Model Adequacy**: Hosmer-Lemeshow test checks if predicted
    probabilities match observed frequencies
4.  **Separation**: Perfect or quasi-perfect separation can cause
    convergence issues and inflated standard errors

**Classification Performance:** Beyond the statistical diagnostics, we
should evaluate practical classification performance: - **Confusion
Matrix**: Shows true vs predicted classifications - **Accuracy**:
Overall percentage of correct predictions  
- **Precision/Recall**: Important when classes are imbalanced -
**ROC/AUC**: Measures discrimination ability across different thresholds

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
# Make predictions on test set
y_pred_proba = results.predict(X_test_sm)
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate classification metrics
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.3f}")
print(f"Confusion Matrix:\n{conf_matrix}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [ ]:
# Create visualization of the logistic regression results
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Data points colored by true class
ax1 = axes[0]
scatter = ax1.scatter(X_train['feature1'], X_train['feature2'], 
                     c=y_train, cmap='RdYlBu', alpha=0.7, s=50)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('Training Data (True Classes)')
plt.colorbar(scatter, ax=ax1)

# Plot 2: Decision boundary and predicted probabilities
ax2 = axes[1]

# Create a mesh to plot the decision boundary
h = 0.1
x_min, x_max = X_train['feature1'].min() - 1, X_train['feature1'].max() + 1
y_min, y_max = X_train['feature2'].min() - 1, X_train['feature2'].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predict probabilities on the mesh
mesh_points = np.c_[xx.ravel(), yy.ravel()]
mesh_points_sm = sm.add_constant(mesh_points)
Z = results.predict(mesh_points_sm)
Z = Z.reshape(xx.shape)

# Plot decision boundary and probability contours
contour = ax2.contourf(xx, yy, Z, levels=50, alpha=0.6, cmap='RdYlBu')
ax2.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

# Plot training points
ax2.scatter(X_train['feature1'], X_train['feature2'], 
           c=y_train, cmap='RdYlBu', edgecolors='black', s=50)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title('Decision Boundary and Probability Contours')
plt.colorbar(contour, ax=ax2)

plt.tight_layout()
plt.show()
mlai.write_figure("logistic-regression-classification-statsmodels.svg", directory="./ml")

Looking at our classification results, we can evaluate several aspects
of model performance:

**Decision Boundary Analysis:** The visualization shows how our logistic
regression model creates a linear decision boundary in the feature
space. The dashed black line represents the 0.5 probability threshold
where the model switches between predicting class 0 and class 1. The
colored contours show the predicted probability landscape - areas closer
to red have higher probability of being class 1, while areas closer to
blue have higher probability of being class 0.

**Model Performance:** From the classification metrics, we can assess: -
**Accuracy**: Overall percentage of correct predictions on the test set
- **Confusion Matrix**: Shows the breakdown of true positives, false
positives, true negatives, and false negatives - **Precision and
Recall**: Important when we care about specific types of errors (e.g.,
medical diagnosis)

**Potential Issues to Monitor:** 1. **Feature Scaling**: If features
have very different scales, consider standardization 2. **Linear
Separability**: Our model assumes a linear decision boundary - if
classes aren’t linearly separable, consider polynomial features or
non-linear methods 3. **Class Imbalance**: If one class dominates,
consider resampling techniques or adjusting the decision threshold 4.
**Overfitting**: Monitor performance on validation data, especially with
many features

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/logistic-regression-classification-statsmodels.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Logistic regression classification results showing training
data and decision boundary with probability contours using
`statsmodels`.</i>

The classification visualization reveals several important aspects of
our logistic regression model:

**Linear Decision Boundary:** The model creates a straight-line decision
boundary (shown as the dashed line at 0.5 probability). This linear
separator works well when classes are roughly linearly separable, but
may struggle with more complex class distributions.

**Probability Gradients:** The colored contours show how predicted
probabilities change smoothly across the feature space. Points far from
the decision boundary have probabilities close to 0 or 1 (high
confidence), while points near the boundary have probabilities around
0.5 (uncertain predictions).

**Model Extensions:** For more complex classification problems, we can
enhance the basic logistic regression model: - **Polynomial Features**:
Add $x_1^2$, $x_2^2$, $x_1 x_2$ terms for non-linear decision boundaries
- **Feature Interactions**: Include products of features to capture
synergistic effects - **Regularization**: Add L1 (Lasso) or L2 (Ridge)
penalties to prevent overfitting - **Feature Engineering**: Transform or
combine features to better capture relationships

To incorporate multiple features and transformations into our model, we
need a systematic way to organize this information through the design
matrix.

### Design Matrix for Logistic Regression

The design matrix in logistic regression works similarly to linear
regression but with some important differences. Each row represents an
observation, and columns represent features. However, the interpretation
of the model changes:

For logistic regression:
$$\text{logit}(p_i) = \log\left(\frac{p_i}{1-p_i}\right) = \boldsymbol{ \Phi}_i \mathbf{ w}$$

where $\boldsymbol{ \Phi}_i$ is the $i$-th row of the design matrix and
$p_i = p(y_i = 1|\mathbf{ x}_i)$.

**Feature Engineering for Classification:** We can enhance the design
matrix with additional transformed features:

-   **Polynomial Features**: $x_1^2$, $x_2^2$ for capturing non-linear
    relationships
-   **Interaction Terms**: $x_1 \times x_2$ for capturing feature
    synergies  
-   **Categorical Encodings**: One-hot encoding for categorical
    variables
-   **Standardized Features**: Z-score normalization for better
    numerical stability

The choice of features in the design matrix directly affects the model’s
ability to capture complex decision boundaries while maintaining
interpretability.

In [ ]:
import statsmodels.api as sm
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# Demo of polynomial logistic regression using python statsmodels.
# Create a more complex non-linear classification dataset
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0, 
                         n_informative=2, n_clusters_per_class=2, 
                         random_state=42)

# Convert to DataFrame for easier handling
df = pd.DataFrame(X, columns=['feature1', 'feature2'])
df['target'] = y

# Split into train and test sets
indices = np.random.permutation(df.shape[0])
num_train = int(np.ceil(df.shape[0]*0.7))
train_indices = indices[:num_train]
test_indices = indices[num_train:]

X_train = df[['feature1', 'feature2']].iloc[train_indices]
y_train = df['target'].iloc[train_indices]
X_test = df[['feature1', 'feature2']].iloc[test_indices]
y_test = df['target'].iloc[test_indices]

# Create polynomial features up to degree 2
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)

# Convert back to DataFrame to see feature names
feature_names = poly_features.get_feature_names_out(['feature1', 'feature2'])
X_train_poly_df = pd.DataFrame(X_train_poly, columns=feature_names)
X_test_poly_df = pd.DataFrame(X_test_poly, columns=feature_names)

# Add constant term to design matrix
X_train_poly_sm = sm.add_constant(X_train_poly_df)
X_test_poly_sm = sm.add_constant(X_test_poly_df)

print("Polynomial features:", feature_names)
print("Design matrix shape:", X_train_poly_sm.shape)

# Fit polynomial logistic regression model
model_poly = sm.Logit(y_train.values, X_train_poly_sm)
results_poly = model_poly.fit()
results_poly.summary()

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
# Compare linear vs polynomial logistic regression performance
from sklearn.metrics import accuracy_score

# Predictions from polynomial model
y_pred_poly_proba = results_poly.predict(X_test_poly_sm)
y_pred_poly = (y_pred_poly_proba > 0.5).astype(int)

# Predictions from simple linear model (for comparison)
X_test_linear_sm = sm.add_constant(X_test)
model_linear = sm.Logit(y_train.values, sm.add_constant(X_train))
results_linear = model_linear.fit(disp=0)
y_pred_linear_proba = results_linear.predict(X_test_linear_sm)
y_pred_linear = (y_pred_linear_proba > 0.5).astype(int)

print(f"Linear Logistic Regression Test Accuracy: {accuracy_score(y_test, y_pred_linear):.3f}")
print(f"Polynomial Logistic Regression Test Accuracy: {accuracy_score(y_test, y_pred_poly):.3f}")

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Create a mesh for decision boundary plotting
h = 0.1
x_min, x_max = X_train['feature1'].min() - 1, X_train['feature1'].max() + 1
y_min, y_max = X_train['feature2'].min() - 1, X_train['feature2'].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Plot 1: Linear logistic regression
ax1 = axes[0]
mesh_points = np.c_[xx.ravel(), yy.ravel()]
mesh_points_sm = sm.add_constant(mesh_points)
Z_linear = results_linear.predict(mesh_points_sm)
Z_linear = Z_linear.reshape(xx.shape)

contour1 = ax1.contourf(xx, yy, Z_linear, levels=50, alpha=0.6, cmap='RdYlBu')
ax1.contour(xx, yy, Z_linear, levels=[0.5], colors='black', linewidths=2)
ax1.scatter(X_train['feature1'], X_train['feature2'], 
           c=y_train, cmap='RdYlBu', edgecolors='black', s=50)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('Linear Logistic Regression')

# Plot 2: Polynomial logistic regression
ax2 = axes[1]
mesh_points_poly = poly_features.transform(mesh_points)
mesh_points_poly_sm = sm.add_constant(mesh_points_poly)
Z_poly = results_poly.predict(mesh_points_poly_sm)
Z_poly = Z_poly.reshape(xx.shape)

contour2 = ax2.contourf(xx, yy, Z_poly, levels=50, alpha=0.6, cmap='RdYlBu')
ax2.contour(xx, yy, Z_poly, levels=[0.5], colors='black', linewidths=2)
ax2.scatter(X_train['feature1'], X_train['feature2'], 
           c=y_train, cmap='RdYlBu', edgecolors='black', s=50)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title('Polynomial Logistic Regression')

plt.tight_layout()
plt.show()
mlai.write_figure("polynomial-logistic-regression-comparison-statsmodels.svg", directory="./ml")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/polynomial-logistic-regression-comparison-statsmodels.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Comparison between linear and polynomial logistic regression
decision boundaries using `statsmodels`.</i>

The comparison between linear and polynomial logistic regression reveals
important insights about model flexibility and performance:

**Linear vs Non-linear Decision Boundaries:** - The linear model (left)
creates a straight decision boundary, which may be too restrictive for
complex class distributions - The polynomial model (right) can create
curved decision boundaries that better separate non-linearly separable
classes - The polynomial features ($x_1^2$, $x_2^2$, $x_1 x_2$) allow
the model to capture quadratic relationships

**Model Performance Trade-offs:** The polynomial model typically shows:
1. **Improved Training Accuracy**: Better fit to training data due to
increased flexibility 2. **Risk of Overfitting**: More parameters may
lead to poor generalization 3. **Interpretability Loss**: Coefficients
for polynomial terms are harder to interpret 4. **Computational
Complexity**: More features require more computation

**Feature Engineering Considerations:** When adding polynomial features,
consider: - **Feature Scaling**: Polynomial features can have very
different scales (e.g., $x$ vs $x^2$) - **Multicollinearity**:
Polynomial features are often highly correlated - **Regularization**:
L1/L2 penalties become more important with many features -
**Cross-validation**: Essential for selecting optimal polynomial degree

The statsmodels summary shows how each polynomial term contributes to
the model, with p-values indicating which transformations are
statistically significant for improving classification performance.

## Other GLMs

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/other-glms-statsmodels.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/other-glms-statsmodels.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We’ve introduced the formalism for generalised linear models. Have a
think about how you might model count data using the [Poisson
distribution](http://en.wikipedia.org/wiki/Poisson_distribution) and a
log link function for the rate, $\lambda(\mathbf{ x})$. If you want a
data set you can try the `pods.datasets.google_trends()` for some count
data.

## Other GLMs

We’ve introduced the formalism for generalised linear models. Have a
think about how you might model count data using the [Poisson
distribution](http://en.wikipedia.org/wiki/Poisson_distribution) and a
log link function for the rate, $\lambda(\mathbf{ x})$. If you want a
data set you can try the `pods.datasets.google_trends()` for some count
data.

## Poisson Distribution

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/poisson-distribution.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/poisson-distribution.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import mlai.plot as plot

In [ ]:
plot.poisson('./ml/')

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/poisson.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>The Poisson distribution.</i>

## Poisson Regression

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/poisson-regression.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/poisson-regression.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Poisson regression is a type of generalized linear model (GLM) used when
modeling count data. It assumes the response variable follows a Poisson
distribution and uses a logarithmic link function to relate the mean of
the response to the linear predictor.

In this model, we make the rate parameter λ a function of covariates
(like space or time). The logarithm of the rate is modeled as a linear
combination of the input features:

$$\log \lambda(\mathbf{ x}, t) = \mathbf{ w}_x^\top \boldsymbol{ \phi}_x(\mathbf{ x}) + \mathbf{ w}_t^\top \boldsymbol{ \phi}_t(t)$$

where: - $\mathbf{ w}_x$ and $\mathbf{ w}_t$ are parameter vectors -
$\boldsymbol{ \phi}_x(\mathbf{ x})$ and $\boldsymbol{ \phi}_t(t)$ are
basis functions for space and time respectively

This formulation is known as a log-linear or log-additive model because
we’re adding terms in the log space. The logarithm serves as our link
function, connecting the linear predictor to the response variable’s
mean.

An important characteristic of this model that practitioners should be
aware of is that while we add terms in the log space, the model becomes
multiplicative when we transform back to the original space. This
happens because:

1.  We start with the log-additive form:
    $\log \lambda(\mathbf{ x}, t) = f_x(\mathbf{ x}) + f_t(t)$

2.  When we exponentiate both sides to get back to λ, the addition in
    log space becomes multiplication:
    $$\lambda(\mathbf{ x}, t) = \exp(f_x(\mathbf{ x}) + f_t(t)) = \exp(f_x(\mathbf{ x}))\exp(f_t(t))$$

This multiplicative nature has important implications for
interpretation. For example, if we increase one input variable, it has a
multiplicative effect on the rate, not an additive one. This can lead to
rapid growth in the predicted counts as input values increase.

Let’s look at another example using synthetic data to demonstrate
Poisson regression without relying on external APIs.

In [ ]:
import numpy as np
import statsmodels.api as sm

In [ ]:
# Generate some example count data
np.random.seed(42)
n_samples = 100
x1 = np.random.uniform(0, 10, n_samples)
x2 = np.random.uniform(0, 5, n_samples)
X = np.column_stack((x1, x2))

# True relationship: y ~ Poisson(exp(1 + 0.3*x1 - 0.2*x2))
lambda_true = np.exp(1 + 0.3*x1 - 0.2*x2)
y = np.random.poisson(lambda_true)

# Fit Poisson regression
model_synthetic = sm.GLM(y, sm.add_constant(X), family=sm.families.Poisson())
result_synthetic = model_synthetic.fit()

The `statsmodels` library in Python provides a convenient way to fit
Poisson regression models. The `sm.GLM` function is used to fit
generalized linear models, and we specify the Poisson family to indicate
that we’re modeling count data. The `sm.add_constant(x)` function adds a
column of ones to the design matrix to account for the intercept term.

In this synthetic example, we generate count data that follows a Poisson
distribution where the rate parameter $\lambda$ depends on two predictor
variables. This demonstrates how Poisson regression can model count data
with multiple predictors.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot actual vs predicted counts
y_pred = result_synthetic.predict(sm.add_constant(X))
ax1.scatter(y, y_pred, alpha=0.5)
ax1.plot([0, max(y)], [0, max(y)], 'r--')
ax1.set_xlabel('Actual Counts')
ax1.set_ylabel('Predicted Counts')
ax1.set_title('Actual vs Predicted')

# Plot residuals
residuals = y - y_pred
ax2.scatter(y_pred, residuals, alpha=0.5)
ax2.axhline(y=0, color='r', linestyle='--')
ax2.set_xlabel('Predicted Counts')
ax2.set_ylabel('Residuals')
ax2.set_title('Residual Plot')

plt.tight_layout()
plt.show()
mlai.write_figure("poisson-regression-diagnostics.svg", directory="./ml/")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/poisson-regression-diagnostics.svg.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Diagnostic plots for the Poisson regression model showing
actual vs predicted counts and residual analysis.</i>

## Practical Tips

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/glm-practical-tips.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/glm-practical-tips.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

When working with generalised linear models in practice, there are
several key considerations that can significantly impact model
performance:

Feature engineering is often the most critical factor in model success:

-   Build modular data processing pipelines that allow you to easily
    test different feature sets. For example, if modeling house prices,
    you might want to test combinations of raw features (square footage,
    bedrooms), derived features (price per square foot), and interaction
    terms (bedrooms × bathrooms).
-   Consider non-linear transformations of continuous variables. For
    instance, taking the log of price data often helps normalize
    distributions.
-   Be thoughtful about encoding categorical variables - one-hot
    encoding isn’t always optimal. For high-cardinality categories,
    consider target encoding or feature hashing.
-   Scale features appropriately - standardization or min-max scaling
    depending on your model assumptions.
-   Document your feature creation process thoroughly, including the
    rationale for each transformation.

Model validation requires careful consideration:

-   Cross-validation should match your real-world use case. For time
    series data, use time-based splits rather than random splits.
-   Bootstrap sampling helps understand parameter uncertainty. For
    example, bootstrapping can show if a coefficient’s sign might flip
    under different samples.
-   Hold-out test sets should be truly independent. In a customer churn
    model, this might mean testing on future customers rather than a
    random subset.
-   Watch for data leakage, especially with time-dependent features. If
    predicting customer churn, using future purchase data would create
    leakage.

Diagnostic checks are essential for model reliability:

-   Create residual plots against fitted values and each predictor. Look
    for systematic patterns - a U-shaped residual plot suggests missing
    quadratic terms.
-   For logistic regression, plot predicted probabilities against actual
    outcomes in bins to check calibration.
-   Calculate influence measures like Cook’s distance to identify
    outliers. In a house price model, a mansion might have outsized
    influence on coefficients.
-   Check Variance Inflation Factors (VIF) for multicollinearity. High
    VIF (\>5-10) suggests problematic correlation between predictors.

Visualization remains crucial throughout:

-   Before modeling, create scatter plots, box plots, and histograms to
    understand your data distribution and relationships.
-   Use pairs plots to identify correlations and potential interactions
    between features.
-   Create residual diagnostic plots including Q-Q plots for normality
    checking.
-   When communicating results, focus on interpretable visualizations.
    For instance, partial dependence plots can show how predictions
    change with a single feature.

Additional practical considerations:

-   Start simple and add complexity incrementally. A basic linear model
    often provides a good baseline.
-   Keep track of model performance metrics across iterations to ensure
    changes actually improve results.
-   Consider the computational cost of feature engineering - some
    transformations might not be feasible in production.
-   Think about how features will be available in production. If a
    feature requires complex processing or external data, it might not
    be practical.
-   For categorical variables with many levels, consider grouping rare
    categories.
-   When dealing with missing data, document your imputation strategy
    and test its impact on model performance.

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

::: {.cell .markdown}

## References